# W1 · K6 — one firewall rule per exe

**Checks:** each exe gets its own firewall rule (`Resonance Stream (Packet Sniffing) [<hash of the exe path>]`),
the old shared rule is gone, and capture works from both exes.

**You need:** the dev exe (`target\debug\resonance-stream.exe`, from `cargo tauri dev`) and the installed/release exe,
and the game. Run Jupyter as Administrator. Run the cells top to bottom; the notebook asks you to do the clicks
and checks the rules itself (PowerShell, so it works on a Korean Windows too).

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "w1" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find test-w1 above it
from w1 import common
from w1.common import Recorder, capture, ask_text
from w1 import firewall
rec = Recorder("w1-firewall")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")

## Which exes
Paste the full path of each exe (Explorer: select the file, *Copy as path*).

In [ ]:
DEV = ask_text("DEV_EXE", "Dev exe path (target\\debug\\resonance-stream.exe): ")
RELEASE = ask_text("RELEASE_EXE", "Installed / release exe path: ")
print("dev     ", DEV, "->", firewall.rule_name_for(DEV))
print("release ", RELEASE, "->", firewall.rule_name_for(RELEASE))

## Step 1 — dev exe: wizard, allow, captured
Start the dev exe (`cargo tauri dev`). If the setup wizard appears, run it and allow. Then log into the game.

In [ ]:
rec.manual("K6-1", "dev exe: wizard (if shown) -> allow -> chat is captured",
           "Start `cargo tauri dev` as Administrator. Wizard -> allow. Open the game; chat lines appear in the app.",
           "chat is captured")

## Step 2 — release exe: the wizard appears once
Close the dev exe first. Start the installed/release exe.

In [ ]:
rec.manual("K6-2", "release exe: wizard appears ONCE -> allow -> chat is captured",
           "Close the dev app. Start the release exe. Its rule does not exist yet, so the wizard should appear; allow it. "
           "Then chat should be captured. Close and start it again: no wizard this time.",
           "wizard once, then chat captured, no wizard on the second start")

## Step 3 — back to dev: no wizard
Close the release exe, start the dev exe again.

In [ ]:
rec.manual("K6-3", "dev exe again: no wizard, still captured",
           "Close the release app. Start the dev app.",
           "no wizard, chat captured")

## Step 4 — the rules themselves (automatic)
Asks Windows for every `Resonance Stream (Packet Sniffing)*` rule and checks them.

In [ ]:
code, text = capture(["powershell", "-NoProfile", "-Command", firewall.POWERSHELL_COMMAND],
                     fixture="firewall_two_rules.json")
if code != 0:
    rec.auto("K6-4", "ask Windows for the rules", False, f"exit {code}: {text[:200]}")
else:
    rules = firewall.parse_rules(text)
    for r in rules:
        print(f"{r.name}\n    {r.program}  enabled={r.enabled} {r.direction} {r.action}")
    for title, ok, evidence in firewall.check_rules(rules, [DEV, RELEASE]):
        rec.auto("K6-4", title, ok, evidence)

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())